# Gradient Boosting Evaluation

This notebook tests whether nonlinear relationships and predictor interactions improve EPL remaining-points forecasts. Experiments A, B, and C use the same predictor progression, validation folds, test seasons, and performance metrics as the primary analysis.

The tree model does not require feature scaling. Hyperparameters controlling tree count, learning rate, depth, leaf size, and subsampling are selected using validation MSE.


## 1. Imports and configuration


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV

RANDOM_STATE = 4031
DEVELOPMENT_SEASONS = list(range(2016, 2023))
TEST_SEASONS = list(range(2023, 2026))

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")


## 2. Load data and define experiments


In [2]:
def locate_dataset(filename="epl_merged_data.xlsx"):
    for root in [Path.cwd(), *Path.cwd().parents]:
        candidate = root / "outputs" / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate outputs/{filename} from {Path.cwd().resolve()}"
    )


dataset_path = locate_dataset()
epl_data = pd.read_excel(dataset_path)

required_columns = {
    "Season", "Team", "Promoted", "Curr_Points", "EOS_Total_Points"
}
missing_required = sorted(required_columns.difference(epl_data.columns))
if missing_required:
    raise KeyError(f"Missing required columns: {missing_required}")
if epl_data.duplicated(["Season", "Team"]).any():
    raise ValueError("The dataset contains duplicate Season-Team observations.")

epl_data["Remaining_Points"] = (
    epl_data["EOS_Total_Points"] - epl_data["Curr_Points"]
)

CURRENT_PERFORMANCE_FEATURES = [
    "Curr_Avg_Win_Percentage",
    "Curr_Avg_Draw_Percentage",
    "Curr_Avg_Goals",
    "Curr_Avg_SOT",
    "Curr_Avg_Opp_Goals",
    "Curr_Avg_Opp_SOT",
    "Curr_Avg_xG",
    "Curr_Avg_xGA",
    "Curr_Avg_Deep",
    "Curr_Avg_Deep_Allowed",
    "Curr_Points",
]
PREVIOUS_SEASON_FEATURES = [
    "Prev_Total_Points",
    "Prev_Avg_Goals",
    "Prev_Avg_SOT",
    "Prev_Avg_Opp_Goals",
    "Prev_Avg_Opp_SOT",
    "Prev_Avg_xG",
    "Prev_Avg_xGA",
    "Prev_Points",
]
TRANSFER_FEATURES = [
    "Fee_Spending",
    "Fee_Earnings",
    "Permanent_In_Count",
    "Permanent_Out_Count",
]

EXPERIMENTS = {
    "Experiment A: Current performance": CURRENT_PERFORMANCE_FEATURES,
    "Experiment B: Current + previous season": (
        CURRENT_PERFORMANCE_FEATURES + PREVIOUS_SEASON_FEATURES + ["Promoted"]
    ),
    "Experiment C: Current + previous + transfers": (
        CURRENT_PERFORMANCE_FEATURES
        + PREVIOUS_SEASON_FEATURES
        + ["Promoted"]
        + TRANSFER_FEATURES
    ),
}
TARGET = "Remaining_Points"

for experiment_name, features in EXPERIMENTS.items():
    missing = sorted(set(features).difference(epl_data.columns))
    if missing:
        raise KeyError(f"{experiment_name} is missing predictors: {missing}")
    if epl_data[features].isna().any().any():
        raise ValueError(f"{experiment_name} contains missing predictor values.")

print(f"Loaded: {dataset_path.resolve()}")
print(f"Observations: {len(epl_data):,}")
for experiment_name, features in EXPERIMENTS.items():
    print(f"{experiment_name}: {len(features)} predictors")


Loaded: C:\Users\micha\epl_results_prediction\outputs\epl_merged_data.xlsx
Observations: 200
Experiment A: Current performance: 11 predictors
Experiment B: Current + previous season: 20 predictors
Experiment C: Current + previous + transfers: 24 predictors


## 3. Create development, validation, and test samples


In [3]:
development = (
    epl_data.loc[epl_data["Season"].isin(DEVELOPMENT_SEASONS)]
    .sort_values(["Season", "Team"])
    .reset_index(drop=True)
)
test_data = (
    epl_data.loc[epl_data["Season"].isin(TEST_SEASONS)]
    .sort_values(["Season", "Team"])
    .reset_index(drop=True)
)

if development.empty or test_data.empty:
    raise ValueError("Development or test data is empty.")

# Grouped leave-one-season-out validation keeps every season intact.
cv_splits = []
for validation_season in DEVELOPMENT_SEASONS:
    train_indices = np.flatnonzero(development["Season"].ne(validation_season))
    validation_indices = np.flatnonzero(
        development["Season"].eq(validation_season)
    )
    if not len(validation_indices):
        raise ValueError(f"No validation observations for season {validation_season}.")
    cv_splits.append((train_indices, validation_indices))

print(f"Development observations: {len(development):,}")
print(f"Validation folds: {len(cv_splits)}")
print(f"Test observations: {len(test_data):,}")


Development observations: 140
Validation folds: 7
Test observations: 60


## 4. Configure gradient boosting


In [4]:
from sklearn.ensemble import GradientBoostingRegressor


gradient_boosting_model = GradientBoostingRegressor(
    loss="squared_error",
    random_state=RANDOM_STATE,
)
parameter_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.03, 0.05],
    "max_depth": [1, 2],
    "min_samples_leaf": [3, 5, 10],
    "subsample": [0.7, 1.0],
}
SCORING = {
    "mse": "neg_mean_squared_error",
    "mae": "neg_mean_absolute_error",
}

combination_count = int(
    np.prod([len(values) for values in parameter_grid.values()])
)
print(f"Gradient boosting hyperparameter combinations: {combination_count}")


Gradient boosting hyperparameter combinations: 48


## 5. Define the evaluation procedure


In [5]:
def evaluate_gradient_boosting(experiment_name, features):
    X_development = development[features]
    y_development = development[TARGET]

    search = GridSearchCV(
        estimator=clone(gradient_boosting_model),
        param_grid=parameter_grid,
        scoring=SCORING,
        refit="mse",
        cv=cv_splits,
        n_jobs=-1,
        error_score="raise",
    )
    search.fit(X_development, y_development)

    fold_metrics = []
    for train_indices, validation_indices in cv_splits:
        fold_model = clone(search.best_estimator_)
        fold_model.fit(
            X_development.iloc[train_indices],
            y_development.iloc[train_indices],
        )
        actual = y_development.iloc[validation_indices]
        predicted = fold_model.predict(X_development.iloc[validation_indices])
        fold_metrics.append(
            {
                "MSE": mean_squared_error(actual, predicted),
                "MAE": mean_absolute_error(actual, predicted),
            }
        )

    fold_metrics = pd.DataFrame(fold_metrics)
    test_actual = test_data[TARGET]
    test_predicted = search.best_estimator_.predict(test_data[features])
    test_mse = mean_squared_error(test_actual, test_predicted)

    return {
        "Experiment": experiment_name,
        "Predictors": len(features),
        "Best_Parameters": search.best_params_,
        "Validation_MSE": fold_metrics["MSE"].mean(),
        "Validation_MAE": fold_metrics["MAE"].mean(),
        "Validation_RMSE": np.sqrt(fold_metrics["MSE"].mean()),
        "Test_MSE": test_mse,
        "Test_MAE": mean_absolute_error(test_actual, test_predicted),
        "Test_RMSE": np.sqrt(test_mse),
    }


## 6. Fit experiments and report performance


In [6]:
gradient_boosting_results = []
for experiment_name, features in EXPERIMENTS.items():
    print(f"Fitting {experiment_name} ...")
    gradient_boosting_results.append(
        evaluate_gradient_boosting(experiment_name, features)
    )

performance_summary = pd.DataFrame(
    [
        {
            "Experiment": result["Experiment"],
            "Predictors": result["Predictors"],
            "Validation_MSE": result["Validation_MSE"],
            "Validation_MAE": result["Validation_MAE"],
            "Validation_RMSE": result["Validation_RMSE"],
            "Test_MSE": result["Test_MSE"],
            "Test_MAE": result["Test_MAE"],
            "Test_RMSE": result["Test_RMSE"],
            "N_Estimators": result["Best_Parameters"]["n_estimators"],
            "Learning_Rate": result["Best_Parameters"]["learning_rate"],
            "Max_Depth": result["Best_Parameters"]["max_depth"],
            "Min_Samples_Leaf": result["Best_Parameters"]["min_samples_leaf"],
            "Subsample": result["Best_Parameters"]["subsample"],
        }
        for result in gradient_boosting_results
    ]
).sort_values("Validation_MSE").reset_index(drop=True)

display(performance_summary)


Fitting Experiment A: Current performance ...


Fitting Experiment B: Current + previous season ...


Fitting Experiment C: Current + previous + transfers ...


,Experiment,Predictors,Validation_MSE,Validation_MAE,Validation_RMSE,Test_MSE,Test_MAE,Test_RMSE,N_Estimators,Learning_Rate,Max_Depth,Min_Samples_Leaf,Subsample
0,Experiment C: Current + previous + transfers,24,84.034,7.433,9.167,96.494,8.189,9.823,100,0.050,1,10,0.700
1,Experiment B: Current + previous season,20,85.344,7.561,9.238,98.026,8.080,9.901,100,0.050,1,10,1.000
2,Experiment A: Current performance,11,87.138,7.597,9.335,100.613,8.183,10.031,100,0.030,1,3,0.700
